# De la lettre au son
### Assembler un phonémiseur, et voir où ça bloque pour le Fongbé

IndabaX Bénin 2026 · Track A1 · Cotonou, 11 septembre

*LAWSON Arnel*
Dépôt : [github.com/Arnel7/fongbe-g2p](https://github.com/Arnel7/fongbe-g2p)

---

### Ce que vous emportez à la fin

1. Un phonémiseur que vous avez assemblé pièce par pièce
2. Une entrée de dictionnaire que vous avez réparée vous-même
3. Un fichier audio produit par votre propre code
4. Une mesure que vous aurez obtenue sur votre écran

Tout tourne sur processeur. Aucune installation sur votre machine.

---

### Lancez la cellule ci-dessous maintenant

Elle prend 2 à 4 minutes. Pendant ce temps, on parle.


In [ ]:
VOIX = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium'

!apt-get -qq install -y espeak-ng
!pip install -q "gruut[fr]==2.4.0" piper-tts numpy transformers fongbe-g2p

!mkdir -p voices
!curl -sL -o voices/fr_FR-siwis-medium.onnx      {VOIX}/fr_FR-siwis-medium.onnx
!curl -sL -o voices/fr_FR-siwis-medium.onnx.json {VOIX}/fr_FR-siwis-medium.onnx.json


### Vérification


In [ ]:
import os

try:
    import gruut, gruut_lang_fr, piper, numpy
except ImportError as e:
    raise SystemExit(f"{e}\n-> Menu Execution > Redemarrer la session, puis relancez cette cellule.")

print('voix :', os.path.getsize('voices/fr_FR-siwis-medium.onnx'), 'octets')
print('Tout est pret.')


---
## Étape 1 · Voir un phonémiseur qui marche · *3 min*

« les poules du couvent couvent »

Le premier `couvent` est un nom, le bâtiment. Le second est un verbe : les poules
couvent leurs œufs. Ils s'écrivent pareil et ne se prononcent pas pareil.

Regardez la colonne du milieu, celle qui donne la nature du mot.

À vous : ajoutez `'les violents violent'` à la liste, puis relancez.


In [ ]:
from gruut import sentences

PHRASE = 'les poules du couvent couvent'

phrases = [
    PHRASE,
    # ajoutez ici : 'les violents violent la loi'
    # (le complement 'la loi' est necessaire : sans lui, gruut etiquette
    #  'violent' en ADJ et l'exemple ne montre plus le contraste)
]

for phrase in phrases:
    print(phrase)
    for sent in sentences(phrase, lang='fr-fr'):
        for mot in sent:
            if mot.phonemes:
                print('   ', mot.text, mot.pos, '->', ''.join(mot.phonemes))
    print()


La nature des mots est juste : `NOUN` puis `VERB`.
La prononciation, non : les deux sortent `kuvɑ̃`, alors que le verbe devrait être `kuv`.

Si vous avez ajouté la seconde phrase : le même défaut, sur un autre mot.

L'étiquetage n'est pas le problème. On revient là-dessus à l'étape 3.


---
## Étape 2 · Nettoyer, puis découper · *4 min*

Ce sont les deux premières pièces du phonémiseur.

### 2a · Écrire les nombres en toutes lettres

À vous : passez `verbaliser` à `True`, puis relancez.


In [ ]:
from gruut import TextProcessor

tp = TextProcessor(default_lang='fr-fr')

texte = 'En 2024, Dupont a payé 1250 euros'
verbaliser = False

graph, root = tp.process(texte, verbalize_numbers=verbaliser)

mots = []
for sent in tp.sentences(graph, root):
    for mot in sent:
        if mot.phonemes:
            mots.append(mot.text)

print(mots)


### 2a-bis - Pourquoi `graph, root` et pas une liste ?

Normaliser transforme les mots : `1250` devient quatre mots. Une liste plate
perdrait le lien avec l'original, alors gruut construit un **arbre**.

- `graph` : l'arbre entier (un `networkx.DiGraph`)
- `root` : le noeud de depart (un objet `SpeakNode`, son numero est `root.node`)

Relancez la cellule ci-dessus avec `verbaliser = True`, puis celle-ci.

Regardez qui n'a **pas** de phonemes : `'1250'`, `'2024'` -- les noeuds parents,
deja remplaces par leurs enfants. **C'est exactement ce que filtre
`if mot.phonemes`** : sans lui, le nombre sortirait deux fois.


In [ ]:
def arbre(n, prof=0):
    d = graph.nodes[n]['data']
    txt = getattr(d, 'text', '') or ''
    ph = getattr(d, 'phonemes', None)
    suite = '  ->  ' + ''.join(ph) if ph else '   (pas de phonemes)'
    print('    ' * prof + f'{type(d).__name__:14} {txt!r:20}{suite}')
    for enfant in graph.successors(n):
        arbre(enfant, prof + 1)

arbre(root.node)


### 2b - Ou coupe-t-on un mot ?

Apostrophes et traits d'union. C'est un vrai probleme, en francais aussi.

Observez : gruut coupe sur les **traits d'union**, mais garde les
**apostrophes** entieres. Ce n'est pas une regle universelle, c'est un choix
de gruut -- et il faudra le refaire pour le Fongbe.

A vous : ajoutez un mot a la liste. Par exemple `"quelqu'un"` ou `"peut-etre"`.


In [ ]:
cas = ["c'est-a-dire", "vas-y", "dis-moi", "l'homme", "aujourd'hui"]

for mot in cas:
    graph, root = tp.process(mot)
    decoupe = []
    for sent in tp.sentences(graph, root):
        for m in sent:
            if m.phonemes:
                decoupe.append(m.text)
    coupe = 'coupe' if len(decoupe) > 1 else 'garde entier'
    print(f'{mot:15} -> {decoupe}   ({coupe})')


---
## Étape 3 · Chercher dans le dictionnaire · *5 min*

### 3a · Le dictionnaire, ouvert

Le dictionnaire de gruut est un simple fichier SQLite. On peut le lire.

À vous : remplacez `'couvent'` par `'violent'`, `'portions'` ou `'content'`.

**Indice** -- verifiez l'ordre avant d'ecrire, il est contre-intuitif :

| `pron_order` | phonemes | sens |
|---|---|---|
| 0 | `p ɔ ʁ s j ɔ̃` | *des* portions (NOM) |
| 1 | `p ɔ ʁ t j ɔ̃` | *nous* portions (VERBE) |


In [ ]:
import sqlite3, os, gruut_lang_fr

chemin = os.path.join(os.path.dirname(gruut_lang_fr.__file__), 'lexicon.db')
con = sqlite3.connect(chemin)

mot = 'couvent'

total = con.execute('SELECT COUNT(DISTINCT word) FROM word_phonemes').fetchone()[0]
multi = con.execute('SELECT COUNT(*) FROM (SELECT word FROM word_phonemes'
                    ' GROUP BY word HAVING COUNT(*) > 1)').fetchone()[0]
avec_role = con.execute("SELECT COUNT(*) FROM word_phonemes"
                        " WHERE role IS NOT NULL AND role != ''").fetchone()[0]

print('mots dans le dictionnaire   :', total)
print('mots à plusieurs prononc.   :', multi)
print('lignes avec un rôle rempli  :', avec_role)

print()
print('Les prononciations de', mot, ':')
for ligne in con.execute('SELECT word, pron_order, phonemes, role'
                         ' FROM word_phonemes WHERE word = ?', (mot,)):
    print('   ', ligne)

con.close()


Les deux prononciations sont déjà là. Le savoir est présent.

Mais la colonne `role`, celle qui dirait *quand* utiliser laquelle, est vide.
Et si vous avez essayé un autre mot : vide aussi. Elle l'est sur toute la base.
Vous venez de le compter.


### 3b · Quand le mot n'est pas dans le dictionnaire

Un nom propre béninois dans une phrase française.


In [ ]:
phrases = ['madame gbaguidi travaille a nyekonakpoe',
           'monsieur kpodjedo habite a cotonou']

for phrase in phrases:
    print(phrase)
    for sent in sentences(phrase, lang='fr-fr'):
        for mot in sent:
            if mot.phonemes:
                print('   ', mot.text, '->', ''.join(mot.phonemes))
    print()


`nyekonakpoe` devient `nikɔnakpo`. Le `ny` se transforme en `ni`, et la syllabe
finale disparaît.

La machine ne produit pas du bruit : elle produit quelque chose qui a l'air correct
et qui ne l'est pas. Personne ne le corrige, parce que personne ne le voit.


### 3c · La réparation

On copie le dictionnaire, et on remplit la colonne `role` dans la copie.
Deux lignes de SQL. Regardez-les : c'est le modèle que vous reproduirez juste après.


In [ ]:
import shutil

source = os.path.dirname(gruut_lang_fr.__file__)
copie = 'gruut_lang_fr_patched'

shutil.rmtree(copie, ignore_errors=True)
shutil.copytree(source, copie)

def patcher(sql):
    con = sqlite3.connect(os.path.join(copie, 'lexicon.db'))
    con.executescript(sql)
    con.commit()
    con.close()

patcher("""
    UPDATE word_phonemes SET role = 'gruut:NOUN'
        WHERE word = 'couvent' AND pron_order = 0;
    UPDATE word_phonemes SET role = 'gruut:VERB'
        WHERE word = 'couvent' AND pron_order = 1;
""")

con = sqlite3.connect(os.path.join(copie, 'lexicon.db'))
for ligne in con.execute("SELECT word, pron_order, phonemes, role"
                         " FROM word_phonemes WHERE word = 'couvent'"):
    print(ligne)
con.close()


Maintenant on charge les deux versions côte à côte, et on compare.


In [ ]:
from gruut.lang import get_settings

avant = TextProcessor(default_lang='fr-fr')
apres = TextProcessor(default_lang='fr-fr',
                      settings={'fr-fr': get_settings('fr-fr', lang_dir=copie)})
# lang_dir = copie : 'apres' lit le dictionnaire CORRIGE,
# 'avant' lit celui installe. Meme code, deux dictionnaires.

def phonemiser(processeur, texte):
    graph, root = processeur.process(texte)
    resultat = []
    for sent in processeur.sentences(graph, root):
        for mot in sent:
            if mot.phonemes:
                resultat.append((mot.text, mot.pos, ''.join(mot.phonemes)))
    return resultat

for (mot, nature, a), (_, _, b) in zip(phonemiser(avant, PHRASE),
                                       phonemiser(apres, PHRASE)):
    print(mot, nature, '|', a, '->', b)
    if a != b:
        print('   corrige')


La nature des mots n'a pas changé. Rien n'a été ajouté au dictionnaire :
`k u v` y était déjà. On a seulement rempli la case qui dit quand l'utiliser.

Une entrée de dictionnaire n'est pas `mot → son`. C'est `(mot, condition) → son`.


### 3d · Réparez un autre mot

`couvent` n'est pas un cas isolé. 1 870 mots ont plusieurs prononciations, et
aucun ne porte la condition qui les départage.

Prenez `portions` :

| phrase | prononciation attendue |
|---|---|
| **nous** portions, le verbe | `pɔʁtjɔ̃` |
| **des** portions, le nom | `pɔʁsjɔ̃` |

En base, `pron_order = 0` est le nom et `pron_order = 1` est le verbe.

À vous : écrivez les deux lignes `UPDATE`, sur le modèle de 3c.


In [ ]:
patcher("""
    -- écrivez ici les deux lignes UPDATE pour 'portions'
""")

apres = TextProcessor(default_lang='fr-fr',
                      settings={'fr-fr': get_settings('fr-fr', lang_dir=copie)})

for mot, nature, ipa in phonemiser(apres, 'nous portions des portions'):
    print(mot, nature, '->', ipa)


Tant que les deux `portions` sonnent pareil, ce n'est pas réparé.
Quand c'est bon, vous lisez `pɔʁtjɔ̃` puis `pɔʁsjɔ̃`.

Le même geste, sur n'importe quel mot. Vous n'avez pas écrit de code : vous avez
ajouté de l'information au dictionnaire.

C'est ce qui manque au Fongbé, sauf qu'il n'y a pas de dictionnaire où l'écrire.


---
## Étape 4 · La machine peut-elle fabriquer le texte ? · *6 min*

À l'étape 3, tout reposait sur un dictionnaire écrit par des humains.
Question naturelle : ne peut-on pas partir du son, et laisser la machine écrire ?

On va le faire, avec votre voix.

La clé est écrite au tableau.


In [ ]:
CLE = ''      # à remplir

BASE = 'https://arnellawson7--indabax-clonage'
TRANSCRIRE = BASE + '-transcription-transcrire.modal.run'
CLONER     = BASE + '-atelier-cloner.modal.run'
FONGBE     = BASE + '-fongbe-transcrire-fongbe.modal.run'

import base64, requests

# les fichiers de secours, si un service ne repond pas
DEPOT = 'https://raw.githubusercontent.com/Arnel7/de-la-lettre-au-son/main'

def envoyer(url, **champs):
    if not CLE:
        raise RuntimeError(
            "La clé n'est pas renseignée. Remontez à la cellule ci-dessus, "
            "mettez-la entre les guillemets de CLE, et relancez-la.")
    audio = open(champs.pop('fichier'), 'rb').read()
    reponse = requests.post(url, timeout=900, json={
        'cle': CLE,
        'audio_b64': base64.b64encode(audio).decode(),
        **champs,
    })
    reponse.raise_for_status()
    return reponse

# Si le réseau ou le service fait défaut, on continue avec ceci.
REPLI_TEXTE = 'Bonjour et bienvenue à IndabaX Bénin.'
REPLI_AUDIO = DEPOT + '/audios/clone-repli.mp3'


### 4a · Un extrait de votre voix

Autorisez le micro quand le navigateur le demande, puis parlez pendant cinq secondes.

Pas de micro ? Passez `MICRO` à `False` : une voix de synthèse sera utilisée.


In [ ]:
MICRO = True

import os
from IPython.display import Audio, display

EXTRAIT = None

if MICRO:
    from google.colab import output
    from IPython.display import Javascript

    display(Javascript("""
    async function enregistrer() {
      const bouton = document.createElement('button');
      bouton.textContent = 'Commencer a parler';
      bouton.style.cssText =
        'font:600 15px system-ui;padding:10px 22px;border:0;border-radius:22px;' +
        'background:#008751;color:#fff;cursor:pointer';
      document.body.appendChild(bouton);
      await new Promise(r => bouton.onclick = r);

      let flux;
      try {
        flux = await navigator.mediaDevices.getUserMedia({audio: true});
      } catch (e) {
        bouton.remove();
        return 'ECHEC:' + e.name;
      }

      const type = MediaRecorder.isTypeSupported('audio/webm;codecs=opus')
                 ? 'audio/webm;codecs=opus' : 'audio/webm';
      const rec = new MediaRecorder(flux, {mimeType: type});
      const morceaux = [];
      rec.ondataavailable = e => { if (e.data.size > 0) morceaux.push(e.data); };

      // on attache onstop AVANT d'appeler stop(), sinon l'evenement peut
      // partir avant que le gestionnaire existe
      const fini = new Promise(r => rec.onstop = r);
      rec.start(250);            // un morceau toutes les 250 ms

      bouton.textContent = 'Arreter';
      bouton.style.background = '#E8112D';
      await new Promise(r => bouton.onclick = r);

      rec.requestData();
      rec.stop();
      await fini;
      flux.getTracks().forEach(t => t.stop());
      bouton.remove();

      const blob = new Blob(morceaux, {type: rec.mimeType});
      if (blob.size === 0) return 'ECHEC:VIDE';
      return await new Promise(r => {
        const lecteur = new FileReader();
        lecteur.onloadend = () => r(lecteur.result);
        lecteur.readAsDataURL(blob);
      });
    }"""))

    try:
        print("Cliquez sur le bouton vert, parlez, puis cliquez sur Arreter.")
        donnees = output.eval_js('enregistrer()', timeout_sec=300)
        if donnees and donnees.startswith('data:'):
            with open('ma_voix.webm', 'wb') as f:
                f.write(base64.b64decode(donnees.split(',')[1]))
            print('recu :', os.path.getsize('ma_voix.webm'), 'octets')
            # MediaRecorder n'ecrit pas la duree dans l'en-tete WebM ; certains
            # decodeurs lisent alors zero echantillon. ffmpeg la reconstruit.
            !ffmpeg -y -loglevel error -i ma_voix.webm -ar 16000 -ac 1 ma_voix.wav
            if os.path.exists('ma_voix.wav') and os.path.getsize('ma_voix.wav') > 4000:
                EXTRAIT = 'ma_voix.wav'
                import wave
                with wave.open(EXTRAIT) as w:
                    print('duree :', round(w.getnframes() / w.getframerate(), 1), 's')
            else:
                print("Le micro n'a rien capte.")
        else:
            print('Micro indisponible :', donnees)
    except Exception as erreur:
        print('Micro indisponible :', type(erreur).__name__)

if EXTRAIT is None:
    print('On continue avec une voix de synthese.')
    !espeak-ng -v fr -w ma_voix.wav "Bonjour et bienvenue a IndabaX Benin"
    EXTRAIT = 'ma_voix.wav'

display(Audio(EXTRAIT))


### 4b · La machine écrit ce qu'elle entend

L'extrait part vers un modèle de reconnaissance vocale.


In [ ]:
try:
    resultat = envoyer(TRANSCRIRE, fichier=EXTRAIT, langue='fr').json()
    TRANSCRIPTION = resultat['texte']
    print('texte  :', TRANSCRIPTION)
    print('langue :', resultat['langue_detectee'], resultat['confiance_langue'])
except Exception as erreur:
    print('Erreur :', erreur)
    print('On continue avec une transcription pré-enregistrée.')
    TRANSCRIPTION = REPLI_TEXTE
    print('texte  :', TRANSCRIPTION)


En français, la machine sait fabriquer le texte. Presque parfaitement.

Corrigez `TRANSCRIPTION` à la main si un mot est faux : la suite s'en sert.


### 4c · Ce texte n'est pas décoratif : sans lui, pas de clonage

On va faire dire à votre voix une phrase que vous n'avez jamais prononcée.

Le modèle a besoin de deux choses : votre extrait, **et sa transcription**.
C'est ce que vous venez d'obtenir.

*Comptez une minute, et ne relancez pas la cellule en boucle.*


In [ ]:
PHRASE = "Les poules du couvent couvent leurs oeufs dans le couvent."

try:
    reponse = envoyer(CLONER, fichier=EXTRAIT,
                      transcription=TRANSCRIPTION,
                      texte=PHRASE,
                      langue='French')
    with open('ma_voix_clonee.wav', 'wb') as f:
        f.write(reponse.content)
    resultat = 'ma_voix_clonee.wav'
except Exception as erreur:
    print('Erreur :', erreur)
    print('Voici un clone pré-enregistré, fait avec la même chaîne.')
    resultat = 'clone_repli.mp3'
    try:
        with open(resultat, 'wb') as f:
            f.write(requests.get(REPLI_AUDIO, timeout=30).content)
    except Exception:
        import subprocess
        resultat = 'clone_repli.wav'
        subprocess.run(['espeak-ng', '-v', 'fr', '-w', resultat,
                        'Les poules du couvent couvent.'])

display(Audio(resultat))


Cinq secondes de voix ont suffi.

Mais regardez ce que la chaîne a consommé : un son **et** son texte.
Retirez la transcription, le modèle refuse. **Le son seul ne suffit jamais.**


### 4d · Et en Fongbé ?

Il existe des systèmes de reconnaissance vocale pour le fongbé, et les meilleurs
sont bons : environ 9 % d'erreur sur les mots. On va en essayer un,
`chrisjay/fonxlsr`.

D'abord son **vocabulaire de sortie** : la liste des symboles qu'il peut écrire.
C'est un fichier de quelques kilo-octets, on peut le lire sans le modèle.


In [ ]:
from transformers import AutoProcessor

processeur = AutoProcessor.from_pretrained('chrisjay/fonxlsr')
vocabulaire = processeur.tokenizer.get_vocab()

print(len(vocabulaire), 'symboles')
print(' '.join(sorted(vocabulaire)))


`á à é è í ì ó ò ú ù`, les accents combinants, les brèves.

**Ce modèle sait écrire des tons.** Reste à savoir s'il le fait.


### Trois phrases, trois sens

Un locuteur a enregistré trois des six lectures de la phrase. Elles sonnent
différemment et veulent dire trois choses différentes.

*Si les enregistrements manquent, la cellule fabrique un son de remplacement.*


In [ ]:
import os, requests
from IPython.display import Audio, display

EXTRAITS = [
    ('fon-1.wav', 'É ɖɔ̀ nú mì', "Il m'a dit"),
    ('fon-3.wav', 'É ɖɔ̀ nú mi', 'Il vous a dit'),
    ('fon-5.wav', 'É ɖɔ̀ nú mǐ', 'Il nous a dit'),
]

for fichier, forme, sens in EXTRAITS:
    if not os.path.exists(fichier):
        with open(fichier, 'wb') as f:
            f.write(requests.get(f'{DEPOT}/audios/fongbe/{fichier}', timeout=60).content)

for fichier, forme, sens in EXTRAITS:
    print(forme, ' ', sens)
    display(Audio(fichier))


### Ce que la machine en écrit


In [ ]:
resultats = []

for fichier, forme, sens in EXTRAITS:
    try:
        fon = envoyer(FONGBE, fichier=fichier).json()
        wsp = envoyer(TRANSCRIRE, fichier=fichier, langue=None).json()
    except Exception as erreur:
        print('Erreur :', erreur)
        break
    resultats.append(fon['texte'])
    print(f'{forme}   {sens}')
    print(f'   Whisper, langue devinée : {wsp["langue_detectee"]} '
          f'({wsp["confiance_langue"]:.2f})   {wsp["texte"]!r}')
    print(f'   ASR fongbé            : {fon["texte"]!r}   '
          f'{fon["marques_de_ton"]} marque(s) de ton')
    print()

if resultats:
    tons = sum(1 for t in resultats if any(c in t for c in "\u0301\u0300\u0306"))
    print(f'{tons} transcription(s) sur {len(resultats)} portent une marque de ton.')


**Trois sons. Trois sens. Combien de transcriptions ?**

Le ton était dans le signal : vous venez de l'entendre. Il n'est pas dans le texte.

Et le modèle *pouvait* l'écrire : son vocabulaire contient les marques. Il a été
entraîné sur du texte, et ce texte ne les portait pas assez.


### Les deux chemins mènent au même endroit

```
                un humain l'écrit  ──►  texte tonné
    du son  ────┤
                une machine le transcrit  ──►  texte sans les tons
```

En français, les deux routes marchent : vous les avez empruntées toutes les deux.

En fongbé, la seconde perd ce qui compte : et pour qu'elle cesse de le perdre, il
faudrait l'entraîner sur du texte tonné. **Que personne n'a écrit en quantité.**

La machine ne remplace pas le travail : elle le suppose déjà fait.


---

*Vous préférez faire tourner le modèle vous-même plutôt que d'appeler un service ?*
*C'est possible : il pèse 1,26 Go, comptez quelques minutes.*

```python
from transformers import Wav2Vec2ForCTC
import soundfile as sf
import torch

modele = Wav2Vec2ForCTC.from_pretrained('chrisjay/fonxlsr')

signal, _ = sf.read('fon-1.wav', dtype='float32')
entree = processeur(signal, sampling_rate=16000, return_tensors='pt')
with torch.no_grad():
    logits = modele(entree.input_values).logits
print(processeur.batch_decode(torch.argmax(logits, dim=-1))[0])
```

La sortie est la même. C'est le même modèle, au même endroit sur Hugging Face.


---
## Étape 5 · Bascule Fongbé, et la mesure · *4 min*

On change de langue. Le phonémiseur français ne sert plus à rien : il faut un
module qui connaisse `gb`, `kp`, `ny`, la nasalisation et les tons.

`fongbe_g2p` fait ça. Il est publié, sous licence MIT.


In [ ]:
from fongbe_g2p import g2p

for mot in ['gbɛtɔ́', 'kpɔ́n', 'Mawu']:
    print(mot, '->', g2p(mot))


Les digraphes deviennent des unités (`ɡ͡b`, `k͡p`), la nasalisation s'applique,
et les tons sortent : `˥` haut, `˧` moyen, `˩` bas, `˩˥` modulé.

**Sur du texte tonné, la chaîne fonctionne.**


### 5a · Une phrase, six sens

Cet exemple vient de *Lire et compter en fongbé pour ceux qui savent lire le
français*, page 15. Ce n'est pas un cas construit : c'est un manuel qui met en
garde contre le problème.

La phrase **E ɖɔ nu mi**, écrite sans les tons, a six lectures possibles.


In [ ]:
FORMES = [
    ('É ɖɔ̀ nú mì', "Il m'a dit"),
    ('È ɖɔ̀ nú mì', "On m'a dit"),
    ('É ɖɔ̀ nú mi', 'Il vous a dit'),
    ('È ɖɔ̀ nú mi', 'On vous a dit'),
    ('É ɖɔ̀ nú mǐ', 'Il nous a dit'),
    ('È ɖɔ̀ nú mǐ', 'On nous a dit'),
]

sorties = {}
for forme, sens in FORMES:
    phonemes = g2p(forme)
    sorties.setdefault(phonemes, []).append(sens)
    print(forme, ' -> ', phonemes, ' ', sens)

print()
print(len(sorties), 'sorties distinctes sur', len(FORMES))


Six formes, six sorties. **Le module les départage toutes**, y compris le ton
modulé `˩˥` de `mǐ`.

Tant que les tons sont écrits, tout va bien.


### 5b · Maintenant, la phrase telle qu'on l'écrit vraiment

Dans la vie courante, on n'écrit pas les accents.


In [ ]:
ORDINAIRE = 'E ɖɔ nu mi'

print(ORDINAIRE, ' -> ', g2p(ORDINAIRE))
print()
print('les six sorties de tout à l\'heure :')
for s in sorties:
    print('   ', s)


Une seule sortie, et **toutes les voyelles en `˧`**.

Les six sens sont devenus un. Le module n'est pas en cause : il lit des marques
de ton qui ne sont pas là.


### 5c · La mesure : vous l'obtenez vous-même

`˧` veut dire « ton moyen ». Mais il veut aussi dire **« aucune information »**,
et rien ne permet de distinguer les deux.

On compte.


In [ ]:
def mesurer(texte):
    total = 0
    moyen = 0
    for phoneme in g2p(texte).split():
        if any(ton in phoneme for ton in '˥˩˧'):
            total += 1
            if '˧' in phoneme and '˩˥' not in phoneme:
                moyen += 1
    return moyen, total

moyen, total = mesurer(ORDINAIRE)
print('forme ordinaire :', moyen, '/', total, 'voyelles en ton moyen =', f'{moyen/total:.0%}')

moyen = total = 0
for forme, sens in FORMES:
    a, b = mesurer(forme)
    moyen += a
    total += b
print('formes tonées   :', moyen, '/', total, 'voyelles en ton moyen =', f'{moyen/total:.0%}')


**À vous :** retirez les accents d'une des six formes dans `FORMES`, relancez la
cellule 5a, et regardez le nombre de sorties distinctes.

Chaque accent retiré fait disparaître une distinction.


Sur cette phrase, tout est perdu.

Sur des corpus réels, la proportion est du même ordre : la majorité des voyelles
d'un texte fongbé ordinaire sort en ton moyen faute de marque écrite.

**Un modèle entraîné là-dessus apprend du bruit tonal.**

Le blocage n'est ni le modèle, ni le code. **C'est le texte.**


---
## Étape 6 · Écouter le résultat · *3 min*

On envoie les phonèmes produits par votre fonction à un modèle qui sait les lire.

Piper attend normalement du texte, et le convertit lui-même avec espeak : qui fait la
même erreur sur `couvent`. On entre donc juste après.

### 6a · Les deux fonctions


In [ ]:
import unicodedata
import wave
import json
import numpy as np
from piper import PiperVoice
from IPython.display import Audio, display

voix = PiperVoice.load('voices/fr_FR-siwis-medium.onnx')
table = json.load(open('voices/fr_FR-siwis-medium.onnx.json'))['phoneme_id_map']

decomposer = False

def en_phonemes(processeur, texte):
    graph, root = processeur.process(texte)
    suite = []
    for sent in processeur.sentences(graph, root):
        for mot in sent:
            if not mot.phonemes:
                continue
            for phoneme in mot.phonemes:
                if decomposer:
                    suite.extend(unicodedata.normalize('NFD', phoneme))
                else:
                    suite.append(phoneme)
            suite.append(' ')
    return suite

def synthetiser(suite, fichier):
    audio = voix.phoneme_ids_to_audio(voix.phonemes_to_ids(suite))
    if audio.dtype != np.int16:
        audio = (audio * 32767).astype(np.int16)
    sortie = wave.open(fichier, 'wb')
    sortie.setnchannels(1)
    sortie.setsampwidth(2)
    sortie.setframerate(voix.config.sample_rate)
    sortie.writeframes(audio.tobytes())
    sortie.close()


### 6b · Écoutez

Écoutez le quatrième mot : `couvent`, le nom.


In [ ]:
suite = en_phonemes(apres, PHRASE)

absents = []
for symbole in suite:
    if symbole not in table:
        absents.append(symbole)

print('phonèmes envoyés :', ''.join(suite).strip())
print('symboles absents de la table du modèle :', absents)

synthetiser(suite, 'essai.wav')
display(Audio('essai.wav'))


### 6c · Qu'avez-vous entendu ?

La voyelle nasale de `couvent` a disparu. Le mot sonne `kuva`, pas `kuvɑ̃`.

Regardez la ligne des symboles absents : **`ɑ̃`**.

La table de ce modèle contient 154 phonèmes. Elle a `ɑ` et le tilde séparément,
jamais `ɑ̃` d'un seul tenant. Piper n'a pas d'identifiant pour ce symbole, alors il le
jette et continue. Sans erreur, sans avertissement.

La solution est de décomposer le caractère en ses deux parties.

À vous : remontez à la cellule 6a, passez `decomposer` à `True`,
relancez 6a puis 6b, et réécoutez.


Vous venez d'entendre ce qui arrive aux tons du Fongbé.

Un symbole absent de la table n'a pas d'identifiant. Il n'est pas mal prononcé : il n'est pas vu. Personne ne le corrige, parce que rien ne signale la perte.


### 6d · Ce que cette table contient, et ce qu'elle ne contient pas


In [ ]:
print('taille de la table :', len(table), 'phonèmes')
print()

print('Sons du Fongbé :')
for symbole in ['ɖ', 'ɲ', 'ŋ', 'ʋ', 'ɸ', 'x']:
    print('   ', symbole, symbole in table)

print()
print('Marques de ton :')
for symbole in ['˥', '˩', '˦', '˨']:
    print('   ', symbole, symbole in table)


La voix française sait prononcer les sons du Fongbé.
Elle ne peut pas porter un ton. Pas mal : pas du tout.


### 6e · Et le modèle qui lit le Fongbé, lui ?

Il existe : `facebook/mms-tts-fon`. Il prend les lettres directement, sans
phonémiseur. Son vocabulaire est un fichier public : on peut le lire sans
télécharger le modèle.


In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('facebook/mms-tts-fon')
vocabulaire = tok.get_vocab()

print(len(vocabulaire), 'symboles')
print(' '.join(sorted(vocabulaire, key=lambda c: vocabulaire[c])))


Regardez bien : l'accent aigu `́` est là. Le modèle **peut** recevoir un ton haut.

Cherchez l'accent grave.


In [ ]:
import unicodedata

for ton, marque in [('haut', '\u0301'), ('bas', '\u0300')]:
    print('ton', ton, ':')
    for voyelle in 'aeiou\u025b\u0254':
        tonee = unicodedata.normalize('NFC', voyelle + marque)
        retenus = [i for i in tok(tonee)['input_ids'] if i != 0]
        print('   ', tonee, '->', len(retenus), 'symbole(s)', retenus)
    print()


Le ton haut passe sur les sept voyelles.

Le ton bas, non : **ça dépend de la voyelle.**

| voyelle | ce que le modèle en garde |
|---|---|
| `è` `ì` | tout : la forme précomposée est dans le vocabulaire |
| `ɛ̀` `ɔ̀` | la voyelle, pas le ton : aucune forme précomposée n'existe |
| `à` `ò` `ù` | **rien du tout** : ni le ton, ni la voyelle |

Ce n'est pas une décision linguistique. C'est la trace du texte sur lequel le
modèle a été entraîné : `è` et `ì` y figuraient, `à` `ò` `ù` non.

**Le vocabulaire d'un modèle est une photographie de son corpus.**

Ce qu'il fait du ton haut qu'il reçoit, en revanche, personne ne peut le savoir :
il faudrait regarder dedans, et il n'y a rien à ouvrir.

**Ouvrir un vocabulaire n'est pas ouvrir un modèle. C'est la porte, pas la pièce.**

### 6f · Le faire parler vous-même

Les deux sons fongbé joués en ouverture viennent de ce modèle. Il pèse 145 Mo et
tourne sur processeur, plus vite que le temps réel. On ne le télécharge pas
pendant l'atelier : voici de quoi le refaire chez vous.

```python
import torch, wave, numpy as np
from transformers import VitsModel, AutoTokenizer

tok = AutoTokenizer.from_pretrained('facebook/mms-tts-fon')
modele = VitsModel.from_pretrained('facebook/mms-tts-fon')
modele.eval()

def dire(phrase, fichier, graine=0):
    torch.manual_seed(graine)          # le modele tire au sort a chaque appel
    with torch.no_grad():
        onde = modele(**tok(phrase, return_tensors='pt')).waveform[0].numpy()
    with wave.open(fichier, 'wb') as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(modele.config.sampling_rate)
        f.writeframes((np.clip(onde, -1, 1) * 32767).astype('<i2').tobytes())
    return round(len(onde) / modele.config.sampling_rate, 2)

print(dire('E ɖɔ nu mi',  'sans-ton.wav'), 's')
print(dire('È ɖɔ̀ nú mi', 'avec-ton.wav'), 's')
```

Écoutez les deux, puis relancez avec `graine=1`, puis `graine=2`.

L'écart entre « avec les tons » et « sans les tons » est du même ordre que
l'écart entre deux tirages du **même** texte. Mesuré : 0,166 à 0,175 contre
0,154 à 0,175. Écrire les tons ne pèse pas plus lourd que le hasard.

---
## Cellule de secours

Bloqué quelque part ? Choisissez l'étape que vous venez de terminer, lancez, et
reprenez le notebook à l'étape suivante. La cellule reconstruit tout ce qui devrait
exister à ce moment-là.


In [ ]:
etape_terminee = 3  #@param [1, 2, 3, 4, 6] {type:"raw"}

import os, shutil, sqlite3, unicodedata, wave
import gruut_lang_fr
from gruut import sentences, TextProcessor
from gruut.lang import get_settings

PHRASE = 'les poules du couvent couvent'
tp = TextProcessor(default_lang='fr-fr')

def phonemiser(processeur, texte):
    graph, root = processeur.process(texte)
    resultat = []
    for sent in processeur.sentences(graph, root):
        for mot in sent:
            if mot.phonemes:
                resultat.append((mot.text, mot.pos, ''.join(mot.phonemes)))
    return resultat

if etape_terminee >= 3:
    source = os.path.dirname(gruut_lang_fr.__file__)
    copie = 'gruut_lang_fr_patched'
    shutil.rmtree(copie, ignore_errors=True)
    shutil.copytree(source, copie)

    con = sqlite3.connect(os.path.join(copie, 'lexicon.db'))
    con.executescript("""
        UPDATE word_phonemes SET role='gruut:NOUN' WHERE word='couvent' AND pron_order=0;
        UPDATE word_phonemes SET role='gruut:VERB' WHERE word='couvent' AND pron_order=1;
        UPDATE word_phonemes SET role='gruut:NOUN' WHERE word='portions' AND pron_order=0;
        UPDATE word_phonemes SET role='gruut:VERB' WHERE word='portions' AND pron_order=1;
    """)
    con.commit()
    con.close()

    avant = TextProcessor(default_lang='fr-fr')
    apres = TextProcessor(default_lang='fr-fr',
                          settings={'fr-fr': get_settings('fr-fr', lang_dir=copie)})

if etape_terminee >= 4:
    import base64, requests

    BASE = 'https://arnellawson7--indabax-clonage'
    TRANSCRIRE = BASE + '-transcription-transcrire.modal.run'
    CLONER     = BASE + '-atelier-cloner.modal.run'
    try:
        CLE                        # déjà saisie plus haut ? on la garde
    except NameError:
        CLE = ''                   # sinon, la clé du tableau

    def envoyer(url, **champs):
        audio = open(champs.pop('fichier'), 'rb').read()
        reponse = requests.post(url, timeout=900, json={
            'cle': CLE,
            'audio_b64': base64.b64encode(audio).decode(),
            **champs,
        })
        reponse.raise_for_status()
        return reponse

    TRANSCRIPTION = 'Bonjour et bienvenue à IndabaX Bénin.'

if etape_terminee >= 6:
    import json
    import numpy as np
    from piper import PiperVoice

    voix = PiperVoice.load('voices/fr_FR-siwis-medium.onnx')
    table = json.load(open('voices/fr_FR-siwis-medium.onnx.json'))['phoneme_id_map']
    decomposer = True

    def en_phonemes(processeur, texte):
        graph, root = processeur.process(texte)
        suite = []
        for sent in processeur.sentences(graph, root):
            for mot in sent:
                if not mot.phonemes:
                    continue
                for phoneme in mot.phonemes:
                    suite.extend(unicodedata.normalize('NFD', phoneme))
                suite.append(' ')
        return suite

    def synthetiser(suite, fichier):
        audio = voix.phoneme_ids_to_audio(voix.phonemes_to_ids(suite))
        if audio.dtype != np.int16:
            audio = (audio * 32767).astype(np.int16)
        sortie = wave.open(fichier, 'wb')
        sortie.setnchannels(1)
        sortie.setsampwidth(2)
        sortie.setframerate(voix.config.sample_rate)
        sortie.writeframes(audio.tobytes())
        sortie.close()

print('Etat de fin d\'etape', etape_terminee, 'reconstruit.')


---
## Pour aller plus loin

En Fongbé, `gb`, `kp` et `ny` sont **une seule unité**, pas deux lettres.
Aucune machine ne le devine.


In [ ]:
def decouper(mot, unites=('gb', 'kp', 'ny')):
    resultat = []
    i = 0
    while i < len(mot):
        for unite in unites:
            if mot[i:i + len(unite)] == unite:
                resultat.append(unite)
                i += len(unite)
                break
        else:
            resultat.append(mot[i])
            i += 1
    return resultat

for mot in ['gbeto', 'kpo', 'nyonu']:
    print(mot)
    print('   naïf  ', list(mot))
    print('   unités', decouper(mot))


---

## Ce que vous venez de faire

| pièce | où |
|---|---|
| Normaliser le texte | étape 2a |
| Découper en mots | étape 2b |
| Chercher dans le dictionnaire | étape 3a |
| Deviner quand le mot est absent | étape 3b |
| Réparer une entrée | étapes 3c et 3d |
| Envoyer les phonèmes à une voix | étape 6 |

Cinq de ces six pièces existent aujourd'hui pour le Fongbé, dans `fongbe_g2p`.

Celle qui manque est la troisième : le dictionnaire.

---

## Ce qui manque, et ce n'est pas du code

Du texte fongbé tonné et validé par des linguistes.
Sans lui, aucun modèle de restauration tonale ne peut être entraîné.

Le Fongbé est décrit, dans des travaux sérieux. Ce qui n'existe pas, c'est une
description de la prononciation **dans un format qu'une machine peut lire** : comme
le fichier SQLite que vous avez ouvert à l'étape 3.

Si vous êtes linguiste, locuteur, ou si vous travaillez avec le CENALA :
c'est là que vous pouvez changer quelque chose.

---

## Pour continuer

| | |
|---|---|
| Le convertisseur graphème-phonème | [github.com/Arnel7/fongbe-g2p](https://github.com/Arnel7/fongbe-g2p) : MIT |
| Le phonémiseur français utilisé ici | [gruut](https://github.com/rhasspy/gruut) : MIT |
| La synthèse vocale | [piper1-gpl](https://github.com/OHF-Voice/piper1-gpl) : GPL-3.0 |
| Les règles du Fongbé | G. Guillet, *Principes de l'écriture et de la lecture de la langue Fon* |

Merci d'être venus.
